# Web demo M1

Chọn runtime T4, chạy từ trên xuống rồi tải ảnh và nhập câu hỏi trên web. Không train, không xuất test. Web dùng proxy riêng của phiên Colab; ngắt runtime sẽ ngừng phục vụ. Notebook 06 giữ kết quả kiểm kỹ thuật.

In [ ]:
import hashlib
import json
import os
import re
import runpy
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = os.environ.get('VQA_PROJECT_ROOT')
SOURCE_RELEASE = ''
SOURCE_SHA256 = ''
WEB_RUN = 'm1_web_01'
PORT = 7860
RUN_WEB = False
WEB_PROCESS = globals().get('WEB_PROCESS')


In [ ]:
from google.colab import drive

if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive', timeout_ms=600000)
if PROJECT_DIR:
    ROOT = Path(PROJECT_DIR).resolve()
else:
    candidates = [Path('/content/drive/MyDrive/Vietnamese_VQA_NLE')]
    candidates += list(Path('/content/drive/.shortcut-targets-by-id').glob('*/Vietnamese_VQA_NLE'))
    roots = {p.resolve() for p in candidates
             if (p / 'reports/m1_selection/frozen_checkpoint3683_v2/selection.json').is_file()}
    if len(roots) != 1:
        raise ValueError('Set PROJECT_DIR to the mounted project folder')
    ROOT = roots.pop()
os.environ['VQA_PROJECT_ROOT'] = str(ROOT)
print('Project:', ROOT)


In [ ]:
if not re.fullmatch(r'source_[A-Za-z0-9_]+', SOURCE_RELEASE or ''):
    raise ValueError('Select a verified source release')
if not re.fullmatch(r'[0-9a-f]{64}', SOURCE_SHA256 or ''):
    raise ValueError('Select its SHA-256')
if not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_-]{0,63}', WEB_RUN):
    raise ValueError('Use a simple new web run name')
bootstrap = ROOT / 'source/bootstrap_cloud_source.py'
if hashlib.sha256(bootstrap.read_bytes()).hexdigest() != (
    '391172ed1fbf39d0eeeef2c9aa3d056d7770085d0a10574d571470d7f2bbef40'
):
    raise ValueError('Bootstrap checksum mismatch')
REPO = Path(subprocess.check_output([
    sys.executable, str(bootstrap), '--archive',
    str(ROOT / 'source/releases' / f'{SOURCE_RELEASE}.zip'),
    '--sha256', SOURCE_SHA256, '--work-root', '/content/vqa_sources',
], text=True).strip())
run_command = runpy.run_path(str(REPO / 'src/vqa_explanation/utils/process.py'))['run_command']
ENV_DIR = Path('/content/vqa_envs') / f'{SOURCE_RELEASE}_demo'
PYTHON = ENV_DIR / 'bin/python'
run_command([sys.executable, str(REPO / 'scripts/setup_cloud_environment.py'),
             '--profile', 'demo', '--environment-dir', str(ENV_DIR)], cwd=REPO)
WEB = runpy.run_path(str(REPO / 'src/vqa_explanation/demo/colab.py'))
print('Source:', SOURCE_RELEASE)


In [ ]:
from google.colab import output

if RUN_WEB:
    if WEB_PROCESS is not None and WEB_PROCESS.poll() is None:
        raise RuntimeError('Web is already running; use the interface below')
    selection_path = ROOT / 'reports/m1_selection/frozen_checkpoint3683_v2/selection.json'
    selection = json.loads(selection_path.read_text('utf-8'))
    expected = {key: selection[key] for key in ('model_id', 'model_revision')}
    expected.update(experiment_id='M1_SFT', adapter_sha256=selection['adapter']['sha256'])
    proxy = output.eval_js('(async () => ({origin: await google.colab.kernel.proxyPort('
                           + str(PORT) + '), frame: window.location.origin}))()')
    adapter = ROOT / 'runs/m1/sft-full_20261007_023531_870900/training/checkpoint-3683'
    COMMAND = [str(PYTHON), '-u', str(REPO / 'scripts/run_demo.py'), '--mode', 'live',
               '--config', str(REPO / 'configs/demo/m1_colab.example.yaml'),
               '--adapter', str(adapter),
               '--host', '127.0.0.1', '--port', str(PORT), '--colab-origin', proxy['origin'],
               '--colab-frame-origin', proxy['frame']]
    print('Loading M1 on the GPU; do not run this cell twice.')
    WEB_PROCESS, WEB_INFO = WEB['start_web'](COMMAND, cwd=REPO,
        output_dir=ROOT / 'demo_runs' / WEB_RUN, port=PORT, expected_identity=expected,
        source_release=SOURCE_RELEASE)
    print('Web ready:', WEB_INFO['identity']['experiment_id'])
    output.serve_kernel_port_as_iframe(PORT, height=1000)
else:
    print('Set RUN_WEB after checking the source and checkpoint paths.')


## Dừng khi dùng xong

Chỉ đổi cờ dưới thành `True` khi muốn tắt web và giải phóng GPU. Log khởi động nằm ở `demo_runs/<WEB_RUN>`; không lưu ảnh tải lên hoặc ghi vào dữ liệu train.

In [ ]:
STOP_WEB = False
if STOP_WEB and WEB_PROCESS is not None:
    WEB['stop_web'](WEB_PROCESS)
    print('Web stopped.')
